# Final Project Seeds

These are starting points, not assignments. A good project usually starts from one of these and gets narrower. You can also bring your own idea. Either way, **talk to me early**.

Each seed lists: the question, who would pay for the answer, **one row of the final dataset**, free data that I've checked is reachable (as of Sep 2026), the method (tools from this class), a dashboard idea, a **minimum viable version** you could finish in about two weeks, stretch goals, and **what the auditors will poke at**.

## Read this first: the standard

**Your repo, dashboard, and analysis will be interrogated by state-of-the-art AI agents**, as well as by me. They will clone your repo, rerun it, recompute your numbers, and look for the problems that sink real research. Build the project so it survives that.

What they will check, on every project:

1. **It reproduces.** `run_all.py` (or the README's steps) rebuilds every table and figure from the raw inputs, on a fresh machine, with no hand edits. Every number in your slides and dashboard comes from a file in `outputs/`.
2. **Every input has a provenance.** For each raw file: where it came from, when you pulled it, and the code (or link) that gets it again.
3. **The unit of observation is right.** Keys are unique (`assert`), and merges use `validate=`. You can say what one row is.
4. **No look-ahead.** At each date, you only use information that was public then. Watch out for restated financials, today's index membership used for past years (**survivorship**), and fiscal-year dates vs. filing dates.
5. **No LLM leakage.** An LLM may already "know" what happened after a headline. Test on dates after the model's training cutoff, or show it doesn't matter.
6. **No p-hacking.** Keep a log (`specs_tried.md`) of every specification you ran, not just the one that worked. Say how many things you tried.
7. **Magnitudes, not just stars.** Is the effect big enough to matter to the person who'd pay for it? After trading costs, if it's a strategy?
8. **Checks by hand.** At least one number in each pipeline step is recomputed independently and matches (like HW6 Step 3).
9. **Honest limits.** A "threats to validity" section in the README. An auditor who finds a problem you already disclosed is much less of a problem.
10. **You understand it.** Every teammate can explain every step, including the parts an agent wrote. Expect questions.

**Red-team yourself first.** Before each checkpoint, run the _Skeptical review_ prompt from the Quick Reference (_Starter prompts for coding agents_, #5) on your own repo, and fix what it finds. The auditors will be at least that skeptical.

## New seeds (Fall 2026)

### 1. Economic links: does a customer's bad news hit its suppliers?

**Question.** When a big customer announces earnings, do its suppliers' stocks react, and do they react *late*?
**Who'd pay:** a portfolio manager, or a supplier's CFO watching concentration risk.
**Row:** customer-announcement × supplier × event day (AR, CAR, the customer's surprise, the share of sales that customer accounts for).
**Data:** 10-K text from EDGAR. Accounting rules require firms to disclose any customer that is ≥10% of revenue, and many name them (that is how the supplier–customer data in the research gets built). An LLM extracts "supplier → major customer" pairs. Earnings dates and surprises come from yfinance (like HW6), and returns from yfinance.
**Method:** LLM extraction → an event study on suppliers around the *customer's* announcement (HW6 machinery), split by the customer's surprise.
**Dashboard:** pick a customer, see its supplier network and their average CAR.
**MVP:** 20 large customers, their disclosed suppliers, 2 years.
**Stretch:** replicate the drift in Cohen & Frazzini (2008, _Economic Links and Predictable Returns_).
**Auditors will poke at:** the LLM's extraction accuracy (hand-check a sample and report a hit rate), name → ticker matching, timing (the 10-K must predate the event), and customers with many suppliers dominating the averages.

### 2. Post-earnings drift in 2026: is there still money on the table?

**Question.** After big earnings surprises, do prices keep drifting for weeks (Bernard & Thomas, 1989)? Is it tradable after costs?
**Who'd pay:** a quant PM.
**Row:** firm-announcement (surprise %, CAR[−1,+1], CAR[+2,+60], size, analyst coverage).
**Data:** yfinance earnings dates and surprises (about 6 years per firm), and returns. Ken French's factors for risk adjustment.
**Method:** HW6's pipeline scaled to the S&P 500, surprise quintiles, a long-short portfolio formed on day +2, and factor alphas.
**Dashboard:** the drift by quintile, with a slider for holding period and costs.
**MVP:** 100 firms, 5 years, quintile CAR plots.
**Stretch:** does drift survive in small caps but not large? Which firms drift most?
**Auditors will poke at:** **survivorship** (today's S&P 500 list applied to past years), the after-close day-0 rule, overlapping holding periods in the t-stats, and costs.

### 3. The disappearing index effect

**Question.** Stocks used to jump when added to the S&P 500 (Shleifer, 1986). Recent work says the jump has mostly vanished (Greenwood & Sammon, _The Disappearing Index Effect_). What does 2020–2026 look like, and does the **announcement** date or the **effective** date matter more?
**Who'd pay:** index funds (they trade at the effective date), and event-driven hedge funds.
**Row:** addition × event day, with both the announcement date and the effective date.
**Data:** S&P Dow Jones press releases (announcement dates, collected with an agent), Wikipedia's constituent table ("Date added"), and yfinance returns.
**Method:** event studies around two day-0s, plus a cross-section of CARs vs. firm size and the share of float held by index funds.
**MVP:** all additions since 2022.
**Stretch:** deletions. Also Nasdaq-100 additions.
**Auditors will poke at:** confusing the two dates, overlapping events (additions announced in batches), and survivorship (deleted firms may be delisted or renamed, so find their returns another way).

### 4. Did academic research kill the anomalies?

**Question.** Do published return predictors stop working after publication (McLean & Pontiff, 2016)? Which ones survive, and does it depend on how hard they are to trade?
**Who'd pay:** a quant research team deciding which signals to keep.
**Row:** anomaly × month (long-short return, pre- or post-publication, signal category).
**Data:** Open Asset Pricing (Chen & Zimmermann): 200+ anomaly portfolios, free (`pip install openassetpricing`). The publication years are in their documentation.
**Method:** the before-and-after decline, regressions of returns on a post-publication dummy, and heterogeneity by category.
**Dashboard:** pick an anomaly, see its cumulative return with the publication date marked.
**MVP:** replicate the average post-publication decline.
**Stretch:** the "data mining beats theory" claim in the seed tab. Rank signals by past returns, adjust for multiple testing, and test out of sample.
**Auditors will poke at:** in-sample vs. out-of-sample periods, the multiple-testing adjustment you chose, and whether the data version you pulled is recorded.

### 5. Tariff exposure: who lost on Liberation Day?

**Question.** On April 2, 2025 (after the close), the US announced sweeping tariffs. Did firms with more import or China exposure, measured from their own 10-K text, fall more?
**Who'd pay:** a strategist, or a CFO benchmarking their own firm's exposure.
**Row:** firm (CAR on Apr 3–4, 2025, an LLM-scored exposure measure, industry, size).
**Data:** 10-Ks filed before April 2025 (EDGAR), and yfinance returns.
**Method:** LLM scoring of exposure (with a rubric and a hand-checked sample), an event study, and a cross-sectional regression of CAR on exposure with industry controls.
**Dashboard:** exposure vs. CAR scatter, filterable by industry.
**MVP:** S&P 500, one exposure score, one event.
**Stretch:** the April 9 pause as a second event. Do the same firms bounce back?
**Auditors will poke at:** LLM leakage (does the model "know" what happened in April 2025?), whether the exposure measure beats simple industry dummies, and the day-0 timing.

### 6. Bank fragility: SVB, and who's next

**Question.** When Silicon Valley Bank failed (March 10, 2023), did banks with more uninsured deposits and larger unrealized securities losses fall more? Which banks look like that **today**?
**Who'd pay:** bank regulators, or bank investors.
**Row:** bank (CAR around March 2023, uninsured deposit share, unrealized losses / equity, size).
**Data:** FFIEC call reports (free bulk download), the NY Fed's CRSP–FRB link (bank ID → stock), and yfinance returns.
**Method:** an event study plus a cross-sectional regression. See Jiang, Matvos, Piskorski & Seru (2023) on mark-to-market losses and uninsured depositors.
**Dashboard:** a "fragility screen" of today's listed banks, using the same measures.
**MVP:** 100 listed banks, one event.
**Stretch:** add First Republic (May 1, 2023) as a second event.
**Auditors will poke at:** the call-report date (the quarter **before** the event), the ID link quality, and small illiquid banks with noisy returns.

### 7. Do analyst upgrades move prices, and does the broker matter?

**Question.** Do stocks react to rating changes (Womack, 1996)? Are some brokers' calls worth more than others?
**Who'd pay:** a buy-side desk deciding whose research to pay for.
**Row:** rating change × event day (broker, from/to grade, AR, CAR).
**Data:** `yf.Ticker(t).upgrades_downgrades` (broker, date, from and to grade, going back years), and returns.
**Method:** an event study by broker and by type of change. Separate changes that arrive with earnings news from those that don't.
**Dashboard:** a broker leaderboard with confidence intervals.
**MVP:** 50 large firms, 5 years, upgrades vs. downgrades.
**Stretch:** do calls that go against the crowd move prices more?
**Auditors will poke at:** confounds (changes issued right after earnings), the timestamps (pre-open vs. after close), and brokers with few calls ranked by noise.

### 8. Following the insiders

**Question.** Do insider **purchases** predict returns? Are "opportunistic" insiders more informative than "routine" ones (Cohen, Malloy & Pomorski, 2012)?
**Who'd pay:** a stock screener product.
**Row:** insider purchase (filing date, insider role, $ size, routine or opportunistic, CAR[+1,+20]).
**Data:** the SEC's structured Form 3/4/5 data sets (quarterly zips, free), and returns.
**Method:** an event study from the **filing** date (the day it's public), and a classification of routine traders by their past timing.
**Dashboard:** today's opportunistic purchases, with the historical CAR for similar trades.
**MVP:** purchases by CEOs and CFOs of S&P 500 firms, 3 years.
**Stretch:** sales are mostly noise. Show it.
**Auditors will poke at:** transaction date vs. filing date (use the filing date), 10b5-1 planned trades, and duplicate filings.

### 9. Crowded trades: what happens when the hedge funds pile in?

**Question.** Do stocks that many institutions bought last quarter do worse afterward? Is "crowding" a risk?
**Who'd pay:** a risk manager at a fund.
**Row:** stock × quarter (number of 13F filers holding it, change in holders, next-quarter return).
**Data:** the SEC's structured Form 13F data sets (quarterly, free).
**Method:** sorts and regressions, with filing lag handled carefully (13Fs are due 45 days after quarter end).
**Dashboard:** a crowding score for any ticker over time.
**MVP:** 3 years, the largest 500 stocks.
**Auditors will poke at:** look-ahead from using quarter-end dates instead of filing dates, CUSIP → ticker mapping, and the size of the 13F files (plan your storage).

### 10. Who are your real competitors? Text-based peers

**Question.** Do a firm's **text-based** peers (Hoberg & Phillips) move with it more than its standard-industry (SIC or GICS) peers? What does that mean for picking comps for WACC or valuation?
**Who'd pay:** a valuation team choosing comps.
**Row:** firm × peer (similarity score, industry-code match, return correlation).
**Data:** the Hoberg-Phillips Data Library (TNIC peers, free), a CIK/ticker link, and yfinance returns.
**Method:** correlations and a comparison of peer sets. Then redo L15's WACC with each peer set and compare.
**Dashboard:** type a ticker, and see both peer sets side by side, with their betas.
**MVP:** 100 firms, one year of peers.
**Auditors will poke at:** the vintage of the peer data vs. the return period (no look-ahead), and ID matching.

### 11. The cost-of-capital dashboard, audited

**Question.** Rebuild industry costs of capital from free data, reproducibly, and compare them to Damodaran's published numbers. Where do you disagree, and why?
**Who'd pay:** any corporate finance team (and every student in a valuation class).
**Row:** industry × year (median beta, D/E, cost of equity, cost of debt, WACC), for yours and Damodaran's.
**Data:** returns (yfinance), EDGAR's XBRL `companyfacts` API for debt and interest, FRED for rates, and Damodaran's free industry data sets.
**Method:** L15's WACC at scale: betas, levering and unlevering, and aggregation.
**Stretch:** firms say they use hurdle rates well **above** their WACC (see Jagannathan et al., 2016, _Why do firms use high discount rates?_). Compare your WACCs to a survey or to disclosed hurdle rates.
**Auditors will poke at:** XBRL tag choices (debt shows up under many tags), unit consistency, and every assumption documented the way your L15 analyst did.

### 12. The Fed-day drift

**Question.** Stocks have historically risen in the 24 hours before scheduled FOMC announcements (Lucca & Moench, 2015). Still true? And can an LLM read the statement's hawkishness better than the 2-year yield's reaction does?
**Who'd pay:** a macro trader.
**Row:** FOMC meeting (pre-announcement return, day-0 return, 2-year yield change, LLM hawkishness score).
**Data:** FOMC calendar and statements (federalreserve.gov), FRED `DGS2`, and yfinance for SPY and sector ETFs.
**Method:** an event study on the index and sectors, and LLM scoring of statements with a rubric.
**MVP:** 2015–2026 meetings.
**Note:** a 2025 team built a Fed-announcement dashboard. See the Hall of Awesomeness, and go further than they did.
**Auditors will poke at:** LLM leakage (the model has read the press coverage of old statements, so test post-cutoff meetings), and the timing (statements come out at 2pm, and daily data can't split before from after).

### 13. Silicon investors: do LLM personas have the disposition effect?

**Question.** Real investors sell winners too early and hold losers too long (Odean, 1998). Do LLM personas? Does it depend on the persona or the model?
**Who'd pay:** a fintech designing nudges, or researchers deciding whether "silicon subjects" are usable.
**Row:** persona × scenario × model (sell or hold, gain or loss, persona traits).
**Data:** you generate it with EDSL (L11's machinery). Budget it the same way HW5 did.
**Method:** a designed experiment: vary gains and losses while holding everything else fixed, and compare to the published human magnitudes.
**Dashboard:** the effect by persona trait and model.
**MVP:** 2 models, 24 personas, 20 scenarios.
**Auditors will poke at:** prompt sensitivity (does rewording flip the result?), whether the model is just reciting the textbook answer, and the cost log.

### 14. The spot bitcoin ETF approval

**Question.** The SEC approved spot bitcoin ETFs on January 10, 2024, after the close. Who won: crypto miners, Coinbase, MicroStrategy (now Strategy), or the ETF sponsors? Was it priced in beforehand?
**Who'd pay:** a crypto-exposed equity investor.
**Row:** firm × event day, plus bitcoin itself.
**Data:** yfinance (stocks and `BTC-USD`, which trades on weekends: careful). The SEC press release fixes the date.
**Method:** event studies with long pre-event windows (this approval was expected for weeks), and a comparison with the ETH ETF approvals later in 2024.
**Auditors will poke at:** 24/7 crypto returns vs. trading-day stock returns, anticipation, and the fake approval tweet from the SEC's hacked X account the day before (a great confound, if you can document it).

### 15. Compute costs: which stocks are exposed to GPU prices? (industry partner: Ornn)

**A real client.** Ornn, a company that builds markets and price benchmarks for GPU compute, will provide data for this project and would be excited to read your report. If you pick this seed, you're writing for a real reader. **Talk to me first**, and I'll set up the data access.
**Question.** Which companies' share prices are most exposed to the cost of compute? Do GPU rental prices and stock returns move together on the same day, and do changes in compute prices **lead or lag** changes in stocks?
**Who'd pay:** Ornn and its customers (anyone hedging compute costs), a PM sizing AI exposure, or a CFO whose margins depend on GPUs.
**Row:** firm × day (stock return, market return, change in the GPU rental price). The output is one row per firm: its compute beta, standard error, lead and lag coefficients, and group (chipmaker, cloud provider, AI-heavy software, no exposure).
**Data:** about two years of daily GPU rental prices from Ornn (check which GPU models, regions, and rental terms are included). Returns from yfinance, and Ken French's factors.
**Data rule: Ornn's data is never posted publicly.** Not in your repo, not in your dashboard, not anywhere else. Keep the raw files out of git (list the folder in `.gitignore`), and get them from me. So the pipeline still runs for the auditors, commit a small **fake** file with the same columns (like `wacc_FAKE.csv` in the L15 template), and have the README say the real data comes from Ornn through me.
**Method:** regress each firm's daily return on the market and on the change in the compute price: a "compute beta," the same idea as L15's market beta. Then add leads and lags of the compute price change, and compare across groups of firms.
**Dashboard:** pick a firm, and see its compute beta with a confidence interval and its lead-lag profile, next to a ranking of the most exposed firms.
**MVP:** 30 firms in clear groups (chipmakers, cloud providers, AI-heavy software, and a control group with no compute exposure), one GPU price series, two years, same-day compute betas.
**Stretch:** Ornn is open to other questions, shaped around your interests and their data: how well **GPU futures** would hedge compute costs, how **auction and order-book** market designs compare, or how differences across **GPU models, regions, and rental terms** affect building a benchmark price.
**Auditors will poke at:** trends that make any two series look related (use changes, not levels), timing (when is the day's GPU price set, relative to the stock market close? And GPU prices may exist on weekends, like bitcoin in seed 14), ranking many firms (the "most exposed" of 100 firms looks exposed by chance, so log every firm and specification in `specs_tried.md`), whether a compute beta is just an AI-sentiment or Nasdaq factor in disguise, a short sample (two years is one regime), and reproducibility with private data (see the data rule above).

## From earlier terms (still good)

These came from past brainstorms. Talk to me for resources.

- **Replicate "Lazy Prices"** (Cohen, Malloy & Nguyen, 2020): firms that change their 10-K language underperform. Compare with Open Asset Pricing's evidence on which factors hold up out of sample.
- **Evaluate our class portfolios:** turnover and tax costs, factor loadings, and alphas "Table 1" style, from this term's contest data.
- **AI narrative forecaster:** optimistic and pessimistic scenarios from a firm's 10-K projections plus its peers' filings.
- **M&A partner finder:** the steps a middle-market M&A shop takes to find acquirers, done with API calls and deep-research agents.
- **AI-enhanced company profile** with data, commentary, and a chatbot for follow-up questions. It has to beat what Yahoo Finance already shows. It's graded on polish and features.
- **8-K and 10-K chatbot:** pull filings and answer questions about them, **with citations to the filing text**.
- **Product launches from 10-K Item 1s:** new sentences year over year as a measure of new products, then analysis.
- **Neural net asset pricing** à la Gu, Kelly & Xiu (2020).
- **LLM trading backtests without look-ahead**, using ChronoGPT (a model trained only on data up to each date).
- **LLM time-series forecasting with text context** (the "Context is Key" benchmark).
- **Cross-sectional ML explainer:** a dashboard that teaches the core concepts, with a demo of the code and a discussion of what it would take to implement.
- **The pipeline-sensitivity dashboard:** how a model's results change as you vary each cleaning and transformation choice.
- **Geography:** sea-level rise and real estate, and GDP from night lights.
- **Learn a "supercharged" finance package** and show it off (see the LeDataSciFi 2025 textbook, sections 9.7–9.8).
- **Find a paper you love and replicate it.** Ask me first whether the data is feasible.